# ₿ Bitcoin: Ciclos Históricos, Halvings y Proyección 2026–2028

**Autor:** Federico Gregori  
**GitHub:** [github.com/blenddzy](https://github.com/blenddzy)  
**Fuentes:** Glassnode, CoinGecko, on-chain public data, Chequeado, CryptoQuant  
**Última actualización:** Junio 2026  

---

## Pregunta central

El precio de Bitcoin tocó un ATH de **$126.198** en octubre de 2025.  
Hoy (junio 2026) cotiza en **~$65.000** — una corrección del **-48%**.

¿Estamos ante el inicio de un bear market prolongado, o el ciclo tiene una segunda pata?

Este análisis responde esa pregunta usando:
1. **Historia de ciclos** (los 4 halvings completos)
2. **Métricas on-chain** (MVRV, NUPL, SOPR, Hash Rate)
3. **Escenarios de proyección** con probabilidades fundamentadas

> ⚠️ **Disclaimer:** Este análisis es educativo. No es consejo de inversión.  
> Bitcoin puede hacer lo que quiera — los patrones históricos no garantizan resultados futuros.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Rectangle
import matplotlib.patheffects as pe
import warnings
warnings.filterwarnings('ignore')

# Paleta
BG    = '#0D1117'; PANEL = '#161B22'; CARD  = '#1C2333'
GOLD  = '#F7931A'; CYAN  = '#38BDF8'; GREEN = '#10B981'
RED   = '#EF4444'; PURP  = '#A855F7'; WHITE = '#F1F5F9'
GRAY1 = '#8B9DB0'; GRAY2 = '#374151'
HALVING_COLORS = [CYAN, GREEN, GOLD, PURP]

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL, 'axes.edgecolor': GRAY2,
    'axes.labelcolor': GRAY1, 'axes.titlecolor': WHITE, 'xtick.color': GRAY1,
    'ytick.color': GRAY1, 'text.color': WHITE, 'grid.color': GRAY2,
    'grid.linestyle': '--', 'grid.alpha': 0.4, 'figure.dpi': 130,
    'font.family': 'DejaVu Sans'
})

print("Setup OK — paleta y librerias listas.")


## 1. Carga de datos

In [ ]:
df_h = pd.read_csv('../data/raw/halving_history.csv')
df_p = pd.read_csv('../data/raw/price_monthly.csv')
df_p['fecha'] = pd.to_datetime(df_p['fecha'])
df_oc = pd.read_csv('../data/raw/onchain_metrics.csv')

print("=== HALVINGS ===")
print(df_h[['numero','fecha','precio_halving','precio_ath','dias_halving_a_ath','retorno_halving_a_ath_pct']].to_string(index=False))
print(f"\n=== PRECIO MENSUAL ===")
print(f"Rango: {df_p['fecha'].min().strftime('%b %Y')} → {df_p['fecha'].max().strftime('%b %Y')}")
print(f"Min: ${df_p['precio_usd'].min():,} | Max: ${df_p['precio_usd'].max():,}")
print(f"\n=== ON-CHAIN (jun 2026) ===")
print(df_oc[['metrica','valor_actual','zona','señal']].to_string(index=False))


## 2. Historia de precio y los 4 halvings

Bitcoin tiene un mecanismo deflacionario programado en su código:  
cada 210.000 bloques (~4 años), la recompensa por minar un bloque se **reduce a la mitad**.

Esto reduce la oferta nueva de BTC en el mercado. Si la demanda se mantiene o crece,  
el precio sube. Eso es la teoría. La historia empírica la valida — con matices.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.semilogy(df_p['fecha'], df_p['precio_usd'], color=GOLD, linewidth=2.2, zorder=4, label='Precio BTC')
ax.fill_between(df_p['fecha'], df_p['precio_usd'], 1, color=GOLD, alpha=0.07)

halvings_info = [
    ('2012-11-28', 'H1 $12',   CYAN),
    ('2016-07-09', 'H2 $650',  GREEN),
    ('2020-05-11', 'H3 $8.7k', GOLD),
    ('2024-04-20', 'H4 $65k',  PURP),
]
for fecha, label, color in halvings_info:
    ax.axvline(pd.Timestamp(fecha), color=color, linewidth=1.8, linestyle='--', alpha=0.8)
    ax.text(pd.Timestamp(fecha), 0.9, label, color=color, fontsize=8, fontweight='bold',
            ha='center', bbox=dict(boxstyle='round,pad=0.2', facecolor=BG, edgecolor=color, linewidth=1))

for fecha, precio, label in [('2013-11-29',1150,'$1.1k'),('2017-12-17',19700,'$19.7k'),
                               ('2021-11-10',69000,'$69k'),('2025-10-06',126198,'$126k')]:
    ax.scatter([pd.Timestamp(fecha)],[precio],color=RED,s=80,zorder=6)
    ax.annotate('ATH '+label,xy=(pd.Timestamp(fecha),precio),xytext=(0,18),
                textcoords='offset points',fontsize=7.5,color=RED,ha='center',fontweight='bold')

ax.axhline(65000, color=CYAN, linewidth=1, linestyle=':', alpha=0.6)
ax.text(df_p['fecha'].max(), 71000, 'Hoy ~$65k', color=CYAN, fontsize=8)
ax.set_title('Bitcoin: historia completa y los 4 halvings (escala logaritmica)', fontsize=13, fontweight='bold')
ax.set_ylabel('Precio USD (log)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}' if x>=1000 else f'${x:.0f}'))
ax.set_ylim(1, 250000)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/01_precio_historico.png', bbox_inches='tight', facecolor=BG, dpi=130)
plt.show()

print("Observacion: escala logaritmica revela que cada ciclo tiene la misma 'forma'")
print("pero con rendimientos decrecientes. En escala lineal, los ciclos 1 y 2 son invisibles.")


## 3. Rendimientos por ciclo — la ley del tamaño

Cada ciclo genera menos % que el anterior. No porque Bitcoin esté fallando,  
sino porque mover un activo de $2T de market cap requiere mucho más capital  
que mover uno de $200M. Es matemática, no debilidad.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('La ley de los rendimientos decrecientes en Bitcoin', fontsize=12, fontweight='bold')

ciclos   = ['Ciclo 1\n(2012-13)', 'Ciclo 2\n(2016-17)', 'Ciclo 3\n(2020-21)', 'Ciclo 4\n(2024-25)']
retornos = [8483, 2931, 691, 94]
dias_ath = [366, 526, 547, 535]
ax1, ax2 = axes

bars1 = ax1.bar(ciclos, retornos, color=HALVING_COLORS, alpha=0.85, width=0.55, edgecolor=BG)
ax1.set_title('% ganancia desde precio del halving al ATH', fontsize=10)
ax1.set_ylabel('Retorno %')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}%'))
ax1.grid(axis='y', alpha=0.4)
for bar, val in zip(bars1, retornos):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
             f'+{val:,}%', ha='center', fontsize=11, fontweight='bold', color=WHITE)
ax1.annotate('Menos % = activo\nmas maduro y grande', xy=(3,94), xytext=(2.1,2500),
             arrowprops=dict(arrowstyle='->', color=GRAY1, lw=1.2),
             color=GRAY1, fontsize=8.5, ha='center')

bars2 = ax2.bar(ciclos, dias_ath, color=HALVING_COLORS, alpha=0.85, width=0.55, edgecolor=BG)
ax2.set_title('Dias desde el halving hasta el ATH del ciclo', fontsize=10)
ax2.set_ylabel('Dias')
ax2.grid(axis='y', alpha=0.4)
for bar, val in zip(bars2, dias_ath):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
             f'{val}d', ha='center', fontsize=11, fontweight='bold', color=WHITE)
ax2.axhline(480, color=GOLD, linestyle='--', linewidth=1.2, alpha=0.7)
ax2.text(3.55, 493, 'Prom.\n480d', color=GOLD, fontsize=7.5, ha='center')

plt.tight_layout()
plt.savefig('../outputs/02_rendimientos_ciclos.png', bbox_inches='tight', facecolor=BG, dpi=130)
plt.show()

# Tabla resumen
print("Tabla resumen de rendimientos:")
print(df_h[['numero','fecha','precio_halving','precio_ath','dias_halving_a_ath','retorno_halving_a_ath_pct','drawdown_post_ath_pct']].to_string(index=False))


## 4. Overlay de ciclos — ¿dónde está el Ciclo 4 respecto a los anteriores?

Normalizando el precio de cada ciclo al valor del día del halving (= 1x),  
se puede comparar la performance relativa de cada ciclo en el mismo eje temporal.

El Ciclo 4 (violeta) es claramente el más débil en términos de multiplicador.  
Pero también muestra algo interesante: aún no convergió a los niveles de corrección  
post-ciclo (que en ciclos anteriores llegaron a 0.1x–0.3x del precio del halving).


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ciclos_overlay = [
    ('Ciclo 1 (2012)', [0,30,60,90,180,270,366,450,500],         [1,1.5,2,3,8,20,96,40,20],                  CYAN,  1.8),
    ('Ciclo 2 (2016)', [0,30,90,180,270,365,526,600,700],         [1,1.1,0.95,1.2,2,3,31,15,8],               GREEN, 1.8),
    ('Ciclo 3 (2020)', [0,30,60,90,180,270,365,450,547,600,700],  [1,1.1,1.3,1.8,2.8,3.5,5.5,6.2,8.3,5,3],  GOLD,  1.8),
    ('Ciclo 4 (2024) ACTUAL', [0,30,90,180,270,365,450,535,580,630], [1,0.98,0.92,0.85,0.9,1.1,1.3,1.9,1.5,1.4], PURP, 2.8),
]
for nombre, dias, mult, color, lw in ciclos_overlay:
    alpha = 1.0 if '2024' in nombre else 0.65
    ax.plot(dias, mult, color=color, linewidth=lw, alpha=alpha, label=nombre, zorder=4 if '2024' in nombre else 3)
    ax.scatter([dias[-1]], [mult[-1]], color=color, s=60, zorder=5)

ax.axvline(480, color=WHITE, linestyle=':', linewidth=1, alpha=0.35)
ax.text(484, 0.4, 'Prom. hist. 480d', color=WHITE, fontsize=7.5, alpha=0.5)
ax.axhline(1, color=GRAY2, linewidth=0.8)
ax.axvspan(530, 640, alpha=0.07, color=PURP)
ax.text(585, 90, 'Zona\nactual', ha='center', fontsize=8, color=PURP)
ax.set_title('Performance post-halving (precio normalizado: 1x = precio del halving)', fontsize=12, fontweight='bold')
ax.set_xlabel('Dias despues del halving')
ax.set_ylabel('Multiplicador')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}x'))
ax.set_xlim(-20, 750); ax.set_ylim(0, 110)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/03_overlay_ciclos.png', bbox_inches='tight', facecolor=BG, dpi=130)
plt.show()

print("Insight clave: en dia ~580 (hoy), ciclos anteriores estaban en:")
print("  Ciclo 1: ~40x  (habia bajado del pico de 96x)")
print("  Ciclo 2: ~15x  (habia bajado del pico de 31x)")
print("  Ciclo 3: ~3x   (habia bajado del pico de 8.3x)")
print("  Ciclo 4: ~1.4x (subio solo 94% desde el halving)")
print("El ciclo 4 es el mas debil en multiplicador. Pero el precio absoluto es el mayor de la historia.")


## 5. Métricas On-Chain — la radiografía de la blockchain

Las métricas on-chain analizan el comportamiento real de los holders y mineros  
leyendo directamente la blockchain, sin depender del precio de mercado.  
Son las métricas que usan los analistas profesionales para evaluar el ciclo.

### Las 4 métricas clave de este análisis:

| Métrica | ¿Qué mide? | Señal actual |
|---------|------------|--------------|
| **MVRV Z-Score** | Si el precio está sobre/sub valuado vs costo histórico de holders | 0.22 → Acumulación |
| **NUPL** | % del mercado con ganancia no realizada | 0.12 → Hope/Fear (no euforia) |
| **SOPR** | Si los que gastan BTC lo hacen con ganancia o pérdida | 0.95 → Vendiendo en pérdida |
| **Hash Rate** | Poder computacional de la red | ATH histórico → Red más fuerte que nunca |


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Metricas On-Chain: que dice la blockchain sobre el ciclo actual?', fontsize=13, fontweight='bold')

# MVRV simulado historico
ax1 = axes[0, 0]
mvrv_vals = np.concatenate([
    3 + 6*np.sin(np.linspace(0, np.pi, 50)),
    -1 + 4.5*np.sin(np.linspace(0, np.pi, 60)),
    -0.5 + 4*np.sin(np.linspace(0, np.pi, 60)),
    -0.5 + 2.5*np.sin(np.linspace(0, np.pi*0.6, 30)),
])
dias_mv = np.arange(len(mvrv_vals))
ax1.fill_between(dias_mv, 3.5, 11, alpha=0.15, color=RED, label='Sobrevaluado (>3.5)')
ax1.fill_between(dias_mv, 2.0, 3.5, alpha=0.10, color=GOLD, label='Caliente (2-3.5)')
ax1.fill_between(dias_mv, -2, 1.0, alpha=0.15, color=GREEN, label='Acumulacion (<1)')
ax1.plot(dias_mv, mvrv_vals, color=CYAN, linewidth=2, zorder=4)
ax1.scatter([len(dias_mv)-1], [0.22], color=GREEN, s=140, zorder=6, label='HOY: 0.22')
ax1.axhline(0.22, color=GREEN, linestyle=':', linewidth=1, alpha=0.7)
ax1.axhline(3.5, color=RED, linestyle='--', linewidth=1, alpha=0.5)
ax1.set_title('MVRV Z-Score — ciclos historicos', fontsize=10)
ax1.set_ylabel('MVRV Z-Score'); ax1.set_ylim(-2, 11)
ax1.legend(fontsize=7.5); ax1.grid(alpha=0.3)

# NUPL gauge
ax2 = axes[0, 1]
for y0, y1, c, lbl in [(-1.0,0.0,RED,'Capitulacion'),(0.0,0.25,PURP,'Hope/Fear'),
                         (0.25,0.5,GOLD,'Optimismo'),(0.5,0.75,GREEN,'Creencia'),(0.75,1.0,RED,'Euforia/TOP')]:
    ax2.barh(0.5, y1-y0, left=y0, height=1.0, color=c, alpha=0.3, edgecolor=BG)
    ax2.text((y0+y1)/2, 0.5, lbl, ha='center', va='center', fontsize=8.5, fontweight='bold', color=WHITE)
ax2.axvline(0.12, color=GREEN, linewidth=4, zorder=5)
ax2.text(0.12, 1.18, 'HOY: 0.12', ha='center', fontsize=10, color=GREEN, fontweight='bold')
ax2.set_xlim(-1, 1); ax2.set_ylim(0, 1.45)
ax2.set_title('NUPL — Net Unrealized Profit/Loss', fontsize=10)
ax2.set_xlabel('Valor NUPL'); ax2.set_yticks([]); ax2.grid(axis='x', alpha=0.3)

# Drawdowns
ax3 = axes[1, 0]
ciclos_dd = ['Ciclo 1\n(2013-15)', 'Ciclo 2\n(2017-18)', 'Ciclo 3\n(2021-22)', 'Ciclo 4\n(actual)']
drawdowns  = [-84, -83, -77, -48]
bars = ax3.bar(ciclos_dd, drawdowns, color=[GRAY1,GRAY1,GRAY1,GOLD], alpha=0.85, width=0.55, edgecolor=BG)
for bar, val in zip(bars, drawdowns):
    ax3.text(bar.get_x()+bar.get_width()/2, val-2, f'{val}%',
             ha='center', va='top', fontsize=11, fontweight='bold', color=WHITE)
ax3.axhline(-80, color=RED, linestyle='--', linewidth=1.2, alpha=0.6)
ax3.text(3.6, -78, 'Prom.\n-80%', color=RED, fontsize=7.5)
ax3.set_title('Drawdowns post-ATH por ciclo', fontsize=10)
ax3.set_ylabel('Correccion desde ATH (%)'); ax3.grid(axis='y', alpha=0.3)

# Escenarios
ax4 = axes[1, 1]
ax4.set_facecolor(CARD); ax4.set_xlim(0,10); ax4.set_ylim(0,3.8); ax4.axis('off')
ax4.text(5, 3.6, 'Escenarios 2026-2027', ha='center', fontsize=12, fontweight='bold', color=WHITE)
for i, (nombre, desc, color, prob) in enumerate([
    ('BAJISTA  15%',  'Ciclo terminado. BTC baja a $28k-40k.',  RED,   0.15),
    ('BASE     50%',  'Nuevo ATH en H2 2026 - H1 2027. $110k-160k.', GOLD, 0.50),
    ('ALCISTA  35%',  'Superciclo ETF+institucional. >$200k.',  GREEN, 0.35),
]):
    y = 2.7 - i*1.05
    ax4.add_patch(FancyBboxPatch((0.2,y-0.38),9.6,0.88,
        boxstyle='round,pad=0.05',facecolor=PANEL,edgecolor=color,linewidth=1.5))
    ax4.add_patch(Rectangle((0.2,y-0.38),9.6*prob,0.88,facecolor=color,alpha=0.12,zorder=3))
    ax4.text(0.5,y+0.12,nombre,fontsize=9.5,fontweight='bold',color=color)
    ax4.text(0.5,y-0.22,desc,fontsize=8,color=GRAY1)

plt.tight_layout()
plt.savefig('../outputs/04_onchain_dashboard.png', bbox_inches='tight', facecolor=BG, dpi=130)
plt.show()

print("Insight mas importante:")
print("  El MVRV Z-Score en 0.22 es consistente con zonas de acumulacion historicas.")
print("  En 2017 y 2021 el MVRV supero 3.5 antes del crash. En 2025 toco maximo ~2.5.")
print("  Eso sugiere que el mercado nunca llego a euforia real en el ciclo 4.")


## 6. Proyección del Ciclo 4 — 2026 a 2028

### Metodología de los escenarios

No es predicción. Es proyección basada en:
- Comportamiento histórico de ciclos anteriores en el mismo momento temporal
- Métricas on-chain actuales vs lecturas en tops/bottoms históricos
- Factores macro nuevos que no existían en ciclos anteriores (ETFs, Strategy, regulación)

### Variables clave a monitorear

| Variable | Señal bajista | Señal alcista |
|----------|---------------|---------------|
| MVRV Z-Score | Supera 3.5 → vender | Se mantiene <2 → acumular |
| Soporte $50k | Rompe → bear market | Rebota → segunda pata |
| ETF flows | Salidas sostenidas | Entradas récord |
| Hash Rate | Cae >20% | Nuevo ATH |
| Halving 5 (2028) | Ya priceado | Activa nuevo ciclo |


In [ ]:
import pandas as pd
fig, ax = plt.subplots(figsize=(15, 6))

eventos = [
    ('2022-11-01', 15460,  'Bottom $15.4k',   GREEN, 'bottom'),
    ('2024-01-11', 45000,  'ETF lanzamiento',  CYAN,  'event'),
    ('2024-04-20', 64968,  'Halving 4 $65k',   PURP,  'halving'),
    ('2025-01-01', 95000,  'Trump gana',        GOLD,  'event'),
    ('2025-10-06', 126198, 'ATH $126k',         RED,   'ath'),
    ('2026-06-27', 65000,  'HOY $65k',          WHITE, 'now'),
]
fechas_e = [pd.Timestamp(e[0]) for e in eventos]
precios_e = [e[1] for e in eventos]

ax.semilogy(fechas_e, precios_e, color=GOLD, linewidth=2.5, zorder=4)
ax.fill_between(fechas_e, precios_e, 10000, color=GOLD, alpha=0.07)
for fecha_s, precio, label, color, tipo in eventos:
    ax.scatter([pd.Timestamp(fecha_s)], [precio], color=color, s=140 if tipo in ('ath','now') else 90, zorder=6)
    offset = 28 if tipo != 'now' else -38
    ax.annotate(label, xy=(pd.Timestamp(fecha_s), precio), xytext=(0, offset),
                textcoords='offset points', ha='center', fontsize=8, color=color, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=color, lw=1) if abs(offset) > 15 else None)

fechas_proy = [pd.Timestamp('2026-06-27'), pd.Timestamp('2026-12-01'),
               pd.Timestamp('2027-06-01'), pd.Timestamp('2028-03-15')]
ax.plot(fechas_proy, [65000,110000,180000,100000], color=GREEN, linewidth=1.8, linestyle='--', alpha=0.75, label='Alcista (35%)')
ax.plot(fechas_proy, [65000,90000,140000,75000],   color=GOLD,  linewidth=2.2, linestyle='--', alpha=0.85, label='Base (50%)')
ax.plot(fechas_proy, [65000,52000,35000,28000],    color=RED,   linewidth=1.8, linestyle='--', alpha=0.65, label='Bajista (15%)')
ax.fill_between(fechas_proy, [65000,52000,35000,28000], [65000,110000,180000,100000], alpha=0.05, color=WHITE)

ax.axvline(pd.Timestamp('2026-06-27'), color=WHITE, linewidth=1.5, linestyle=':', alpha=0.45)
ax.text(pd.Timestamp('2026-09-01'), 11500, 'Real  |  Proyeccion', fontsize=8.5, color=GRAY1, style='italic')
ax.axvline(pd.Timestamp('2028-03-15'), color=PURP, linewidth=1.5, linestyle='--', alpha=0.5)
ax.text(pd.Timestamp('2028-03-15'), 220000, 'Halving 5\nmar 2028', fontsize=8, color=PURP, ha='center')

ax.set_title('Ciclo 4 de Bitcoin: historia real + proyeccion de escenarios (2026-2028)', fontsize=12, fontweight='bold')
ax.set_ylabel('Precio USD (escala log)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))
ax.set_ylim(10000, 300000)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/05_ciclo4_proyeccion.png', bbox_inches='tight', facecolor=BG, dpi=130)
plt.show()


## 7. Conclusiones

---

### ¿El Ciclo 4 terminó?

**Probablemente no — pero fue estructuralmente diferente.**

El mercado nunca llegó a euforia real (MVRV <2.5 vs umbral histórico de 3.5+).  
Eso puede explicarse por dos razones no excluyentes:

1. **El activo maduró.** Los ETFs y los grandes holders institucionales amortiguan la volatilidad.  
   Menos euforia = ciclos más moderados pero más estables.

2. **El ciclo todavía no terminó.** La segunda pata puede venir en H2 2026 o H1 2027,  
   consistente con el timing histórico (ATH suele llegar 12–18 meses después del halving).

---

### Los 3 niveles de precio a monitorear

| Nivel | Precio | Significado |
|-------|--------|-------------|
| **Soporte crítico** | ~$47.000 | Precio realizado. Si lo rompe, bear market confirmado |
| **Zona neutral** | $65.000–$80.000 | Donde estamos. Acumulación o distribución |
| **Resistencia clave** | $126.198 | ATH anterior. Superarlo confirma nueva fase alcista |

---

### Lo que este análisis NO puede hacer

- Predecir exactamente cuándo ocurrirá el próximo movimiento
- Garantizar que los patrones históricos se repitan
- Ignorar que Bitcoin sigue siendo un activo de alto riesgo y alta volatilidad

**Solo 4 ciclos completos es muy poco para afirmar que un patrón es ley.**

---

*Análisis completo: github.com/blenddzy/bitcoin-cycles | Junio 2026*
